<a href="https://colab.research.google.com/github/albijanashala/ML1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [12]:
DECISION_DATE = "2026-02-28"

features = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS imp_feb,
               SUM(gsc_clicks) AS clicks_feb,
               SUM(gsc_sum_position) AS sum_pos_feb,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_visible_feb,
               MIN(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS best_pos_feb,
               SUM(CASE WHEN report_date <= DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h1,
               SUM(CASE WHEN report_date >  DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h2,
               MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS has_ga4
        FROM {FEB}
        GROUP BY 1, 2
    ),
    mar AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS imp_mar
        FROM {MAR}
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        -- Volume, February only
        f.imp_feb,
        f.clicks_feb,
        f.days_visible_feb,

        -- Rates, both inputs from February
        ROUND(f.clicks_feb::FLOAT / NULLIF(f.imp_feb, 0), 5) AS ctr_feb,
        ROUND(f.imp_feb::FLOAT / NULLIF(f.days_visible_feb, 0), 2) AS imp_per_active_day,

        -- Position: impression-weighted, lower is better
        ROUND(f.sum_pos_feb::FLOAT / NULLIF(f.imp_feb, 0), 2) AS pos_feb,
        COALESCE(f.best_pos_feb, 100) AS best_pos_feb,

        -- Within-February trend: second half against first half
        ROUND(f.imp_h2::FLOAT / NULLIF(f.imp_h1, 0), 3) AS trend_within_feb,

        -- GA4 presence as a flag, since only a small share of rows have GA4
        f.has_ga4,

        -- Content metadata, known before the decision date
        COALESCE(d.word_count, 0) AS word_count,
        CASE WHEN d.word_count IS NULL THEN 1 ELSE 0 END AS word_count_missing,
        COALESCE(d.search_volume, 0) AS search_volume,
        COALESCE(d.competition, 0) AS competition,
        COALESCE(d.backlinks, 0) AS backlinks,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,

        -- Categoricals, filled explicitly
        COALESCE(d.content_type, 'unknown') AS content_type,
        COALESCE(d.competition_level, 'unknown') AS competition_level,
        COALESCE(d.main_intent, 'unknown') AS main_intent,

        -- Label window: March only, never a feature
        COALESCE(m.imp_mar, 0) AS imp_mar

    FROM feb f
    LEFT JOIN mar m USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE f.imp_feb >= 50
    ORDER BY f.client_hash_id, f.content_hash_id
""").df()

print(f"Rows: {len(features):,}")
print(f"Columns: {len(features.columns)}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 93,654
Columns: 21


,client_hash_id,content_hash_id,imp_feb,clicks_feb,days_visible_feb,ctr_feb,imp_per_active_day,pos_feb,best_pos_feb,trend_within_feb,...,word_count,word_count_missing,search_volume,competition,backlinks,content_age_days,content_type,competition_level,main_intent,imp_mar
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,246.0,1.0,28,0.00407,8.79,17.24,8.50,0.662,...,3168,0,0,0.00,0,144,keyword article,LOW,informational,331.0
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,137.0,0.0,27,0.00000,5.07,9.66,7.75,1.076,...,4135,0,20,0.03,9,144,keyword article,LOW,commercial,33.0
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,121.0,0.0,28,0.00000,4.32,9.12,5.75,0.833,...,3211,0,0,0.00,0,144,keyword article,LOW,informational,145.0
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,174.0,0.0,12,0.00000,14.50,13.38,9.00,NaN,...,3465,0,0,0.00,0,144,keyword article,LOW,informational,461.0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,164.0,0.0,28,0.00000,5.86,11.48,7.00,0.451,...,3149,0,0,0.00,0,144,keyword article,LOW,informational,232.0


In [13]:
import numpy as np

df = features.copy()

# February has 28 days, March has 31 — compare daily rates, not raw sums
FEB_DAYS, MAR_DAYS = 28, 31
imp_rate_feb = df["imp_feb"] / FEB_DAYS
imp_rate_mar = df["imp_mar"] / MAR_DAYS

df["is_declining"] = (imp_rate_mar < 0.8 * imp_rate_feb).astype(int)

print(f"Positive rate: {df['is_declining'].mean():.1%} ({df['is_declining'].sum():,} declining)")
print(f"\nMissing values before fill:\n{df.isna().sum()[df.isna().sum() > 0]}")

# Remaining fills, stated explicitly rather than dropped
df["pos_feb"] = df["pos_feb"].fillna(100)
df["no_h1_impressions"] = df["trend_within_feb"].isna().astype(int)
df["trend_within_feb"] = df["trend_within_feb"].fillna(1.0)
df["content_age_days"] = df["content_age_days"].fillna(-1)

print(f"\nRows after fills: {len(df):,} (no rows dropped)")

Positive rate: 27.6% (25,887 declining)

Missing values before fill:
trend_within_feb    12369
dtype: int64

Rows after fills: 93,654 (no rows dropped)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.